In [ ]:
"""
Note: Parameters, credentials, and assets have been modified for intellectual property protection. 
The core logic, including stateful memory management and the daily killswitch, remains intact for architectural review.
"""

import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import pytz

print("Initializing Data Ingestion Engine (Secure Pagination Strategy)...")

# 1. MetaTrader 5 Connection (Sanitized)
if not mt5.initialize(login=99999999, password="YOUR_PASSWORD", server="YOUR_SERVER"):
    print("MT5 Initialization failed.")
    mt5.shutdown()

timezone = pytz.timezone("Etc/UTC")
target_start_date = datetime(2020, 1, 1, tzinfo=timezone)
symbols = ["ASSET_A", "ASSET_B"] # Sanitized Assets
timeframe = mt5.TIMEFRAME_M5
chunk_size = 50000 
dataframes = {}

print(f"Starting massive chunked extraction until {target_start_date.date()} or server limit...")

# 3. Secure Chunked Extraction Engine
for sim in symbols:
    mt5.symbol_select(sim, True)
    chunk_df_list = []
    start_pos = 0
    target_reached = False
    
    print(f"\nDownloading {sim} in progress...")
    
    while not target_reached:
        rates = mt5.copy_rates_from_pos(sim, timeframe, start_pos, chunk_size)
        
        if rates is None or len(rates) == 0:
            print(f"End of data reached at position {start_pos}.")
            break
            
        # Creating a secure dataframe for this chunk
        df_chunk = pd.DataFrame(rates)
        chunk_df_list.append(df_chunk)
        
        oldest_date = pd.to_datetime(rates[0]['time'], unit='s', utc=True)
        print(f"   -> Extracted {len(rates)} candles. History reached: {oldest_date.date()}")
        
        if oldest_date <= target_start_date:
            target_reached = True
        else:
            start_pos += chunk_size
            # If the broker returns a partial chunk, extraction is finished
            if len(rates) < chunk_size:
                print("Database floor reached.")
                break

    if not chunk_df_list:
        print(f"Extraction failed for {sim}.")
        continue
        
    # Concatenating all dataframes without generating KeyError
    df = pd.concat(chunk_df_list, ignore_index=True)
    df['time'] = pd.to_datetime(df['time'], unit='s', utc=True)
    df.set_index('time', inplace=True)
    
    # Cleaning: removing duplicates and sorting chronologically
    df = df[~df.index.duplicated(keep='first')]
    df.sort_index(inplace=True)
    
    # Filtering to target date (if provided by broker)
    df = df[df.index >= target_start_date]
    
    # Standardizing column naming
    dataframes[sim] = df[['close']].rename(columns={'close': f'close_{sim[:3]}'})
    print(f"{sim} Completed: {len(df)} net candles downloaded.")

mt5.shutdown()

# 4. Synchronization and Split
if len(dataframes) < 2:
    print("\nIncomplete extraction.")
else:
    print("\nSynchronizing price matrices (Outer Join)...")
    df_master = dataframes[symbols[0]].join(dataframes[symbols[1]], how='outer')
    df_master.dropna(inplace=True)

    # 5. INSTITUTIONAL SPLIT: 70/30 PERCENTAGE
    print("\nExecuting Institutional Split (70/30)...")
    split_index = int(len(df_master) * 0.70)
    
    df_in_sample = df_master.iloc[:split_index]
    df_out_of_sample = df_master.iloc[split_index:]

    # 6. Saving (Sanitized Paths)
    path_is = 'Asset_A_Asset_B_In_Sample.csv'
    path_oos = 'Asset_A_Asset_B_Out_Of_Sample.csv'
    
    df_in_sample.to_csv(path_is)
    df_out_of_sample.to_csv(path_oos)

    print(f"\nEXTRACTION AND SPLIT COMPLETED SUCCESSFULLY!")
    print(f"IN-SAMPLE (Training - 70%): {len(df_in_sample)} candles (From {df_in_sample.index[0].date()} to {df_in_sample.index[-1].date()})")
    print(f"OUT-OF-SAMPLE (Blind Test - 30%): {len(df_out_of_sample)} candles (From {df_out_of_sample.index[0].date()} to {df_out_of_sample.index[-1].date()})")